# 方法與屬性

## 學習目標

- 分辨三種方法：實例方法、類別方法、靜態方法
- 學會判斷何時用哪一種
- 深入理解 `@property` 的進階用法
- 認識方法鏈接（method chaining）模式

---

## 三種方法一覽

In [ ]:
class Pizza:
    _price_per_inch = 15    # 每吋單價

    def __init__(self, name, size):
        self.name = name
        self.size = size

    # 實例方法：操作個別實例
    def price(self):
        return self.size * self._price_per_inch

    # 類別方法：操作類別本身，常用於替代建構式
    @classmethod
    def margherita(cls, size):
        return cls("瑪格麗特", size)

    # 靜態方法：不需要存取實例或類別，只是邏輯上歸屬這個類別
    @staticmethod
    def is_valid_size(size):
        return size in [6, 9, 12]

### 使用方式

In [ ]:
# 實例方法：透過實例呼叫
p = Pizza("夏威夷", 12)
print(p.price())             # 180

# 類別方法：透過類別呼叫（也能透過實例）
p2 = Pizza.margherita(9)
print(p2.name, p2.price())   # 瑪格麗特 135

# 靜態方法：透過類別或實例都行
print(Pizza.is_valid_size(12))  # True
print(Pizza.is_valid_size(7))   # False

---

## 三種方法比較表

| 特性 | 實例方法 | 類別方法 | 靜態方法 |
|------|----------|----------|----------|
| 裝飾器 | 不需要 | `@classmethod` | `@staticmethod` |
| 第一個參數 | `self`（實例） | `cls`（類別） | 無 |
| 能存取實例屬性？ | 可以 | 不行 | 不行 |
| 能存取類別屬性？ | 可以 | 可以 | 要自己寫類別名 |
| 常見用途 | 操作實例資料 | 替代建構式、工廠方法 | 工具函式 |

### 怎麼選？

```
需要存取 self（實例資料）？
├─ 是 → 實例方法
└─ 否 → 需要存取 cls（類別本身）？
         ├─ 是 → @classmethod
         └─ 否 → @staticmethod
```

---

## 類別方法的經典用途：替代建構式

一個類別可能有多種建立方式：

In [ ]:
class Date:
    def __init__(self, year, month, day):
        self.year = year
        self.month = month
        self.day = day

    @classmethod
    def from_string(cls, date_str):
        """從字串建立：'2024-01-15'"""
        year, month, day = map(int, date_str.split("-"))
        return cls(year, month, day)

    @classmethod
    def today(cls):
        """從今天的日期建立"""
        import datetime
        t = datetime.date.today()
        return cls(t.year, t.month, t.day)

    def __repr__(self):
        return f"{self.year}-{self.month:02d}-{self.day:02d}"

d1 = Date(2024, 1, 15)
d2 = Date.from_string("2024-06-30")
d3 = Date.today()
print(d1, d2, d3)

---

## @property 進階用法

### 唯讀計算屬性

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height

    @property
    def perimeter(self):
        return 2 * (self.width + self.height)

r = Rectangle(3, 4)
print(r.area)        # 12（看起來像屬性，其實是計算出來的）
print(r.perimeter)   # 14

### 快取計算屬性

如果計算很花時間，可以快取結果：

In [ ]:
class DataSet:
    def __init__(self, numbers):
        self._numbers = list(numbers)
        self._stats_cache = None

    @property
    def stats(self):
        if self._stats_cache is None:
            # 假設這是一個耗時的計算
            self._stats_cache = {
                "mean": sum(self._numbers) / len(self._numbers),
                "max": max(self._numbers),
                "min": min(self._numbers),
            }
        return self._stats_cache

ds = DataSet([1, 2, 3, 4, 5])
print(ds.stats)   # 第一次：計算並快取
print(ds.stats)   # 第二次：直接回傳快取

> **提示：** Python 3.8+ 可以用 `functools.cached_property`，效果相同但更簡潔。

---

## 方法鏈接（Method Chaining）

讓方法回傳 `self`，就能串接呼叫：

In [ ]:
class Query:
    def __init__(self, table):
        self._table = table
        self._conditions = []
        self._limit = None

    def where(self, condition):
        self._conditions.append(condition)
        return self     # 回傳自己，讓下一個方法接著呼叫

    def limit(self, n):
        self._limit = n
        return self

    def to_sql(self):
        sql = f"SELECT * FROM {self._table}"
        if self._conditions:
            sql += " WHERE " + " AND ".join(self._conditions)
        if self._limit:
            sql += f" LIMIT {self._limit}"
        return sql

# 串接呼叫，讀起來很流暢
query = (
    Query("students")
    .where("grade > 80")
    .where("age < 20")
    .limit(10)
    .to_sql()
)
print(query)
# SELECT * FROM students WHERE grade > 80 AND age < 20 LIMIT 10

---

## 練習題

寫一個 `Counter` 類別：

1. `__init__` 接收起始值（預設 0）
2. 實例方法 `increment(n=1)` 和 `decrement(n=1)`，支援方法鏈接
3. 類別方法 `from_string(s)` 能從字串建立，例如 `"42"`
4. 靜態方法 `is_valid(value)` 檢查是否為非負整數
5. `@property value` 唯讀，回傳目前的計數

In [ ]:
# 預期結果
c = Counter.from_string("10")
c.increment(5).increment(3).decrement(2)
print(c.value)                # 16
print(Counter.is_valid(-1))   # False

<details>
<summary>參考解答</summary>

In [ ]:
class Counter:
    def __init__(self, start=0):
        self._count = start

    @property
    def value(self):
        return self._count

    def increment(self, n=1):
        self._count += n
        return self

    def decrement(self, n=1):
        self._count -= n
        return self

    @classmethod
    def from_string(cls, s):
        return cls(int(s))

    @staticmethod
    def is_valid(value):
        return isinstance(value, int) and value >= 0

</details>

---

## 本節重點

| 概念 | 說明 |
|------|------|
| 實例方法 | 最常見，透過 `self` 存取實例資料 |
| `@classmethod` | 第一個參數是 `cls`，常用於替代建構式 |
| `@staticmethod` | 不需要 `self` 或 `cls`，邏輯上屬於該類別的工具函式 |
| `@property` | 把方法偽裝成屬性，支援唯讀、計算、快取 |
| 方法鏈接 | 方法回傳 `self`，讓呼叫可以串接 |

---

> **本章結束。** 接下來可以繼續學習繼承與多型等進階主題。